# E5b - Anchor-distance robustness sweep (reproducer)

Top-to-bottom reproducer for `docs/experiments/E5b-anchor-distance-design.md`.

Headline metric: **paired conditional adoption** (M1 paired definition, denominator excludes case 4 `base=a=pred`).

Key finding: adoption is **uncertainty-modulated AND plausibility-windowed**.
- Correct-base records show essentially no anchor effect (~0.01-0.10 across all distances).
- Wrong-base records show sharp plausibility-windowed pattern: adoption peaks at S2 [2,5] for VQAv2 and decays to ~0 at S4/S5 in both datasets.

All heavy lifting in `scripts/analyze_e5b_distance.py` - this notebook just invokes it and displays.

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'scripts'))
from analyze_e5b_distance import run
out = run()
summary = out['summary']
print(f"loaded {out['n_records']} records, wrote {out['out_csv']}")

## Per-cell summary

Rows: (dataset, stratum, base) where base is 'correct' or 'wrong'. `adopt_cond = case2 / (case1+case2+case3)` — fraction of eligible records where the model moved to the anchor.

In [ ]:
import pandas as pd
pd.set_option('display.float_format', '{:0.4f}'.format)
summary

## Headline figure - adoption vs distance, base correctness split

Two panels (VQAv2, TallyQA). Each panel has two lines: correct-base (flat) and wrong-base (peaked at S2).

In [ ]:
from IPython.display import Image, display
display(Image(filename=str(ROOT / 'docs' / 'figures' / 'E5b_adopt_cond_curve.png')))

## Cross-dataset overlay (wrong-base only)

Both datasets show the same plausibility-windowed adoption pattern when the model is uncertain.

In [ ]:
display(Image(filename=str(ROOT / 'docs' / 'figures' / 'E5b_adopt_cond_overlay.png')))

## Sanity check - wrong-base S1 vs S5 ratio

If plausibility-windowed adoption is real, the wrong-base S1/S2 cells should be much higher than S4/S5.

In [ ]:
for ds in ['VQAv2', 'TallyQA']:
    cell = summary[(summary['dataset']==ds) & (summary['base']=='wrong')].set_index('stratum')
    s1 = cell.loc['S1', 'adopt_cond']
    s5 = cell.loc['S5', 'adopt_cond']
    ratio = s1 / s5 if s5 > 0 else float('inf')
    print(f'{ds:<8s} wrong-base adopt_cond: S1={s1:.4f}, S5={s5:.4f}, S1/S5 ratio={ratio:.1f}')